# BERT-Based Fake News Detection

This notebook trains a BERT model to classify news articles as real or fake using **preprocessed data** from `data/preprocessed_data.csv`.

The preprocessed data includes:
- Text normalization (lowercase, URL/HTML removal)
- Number removal
- Tokenization and stopword removal
- Porter stemming

## Part 1: Setup and Data Loading

In [2]:
import bert_utils
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

## Part 2: Load Real News Data

In [7]:
preprocessed_path = 'data/preprocessed_data.csv'
df = pd.read_csv(preprocessed_path)

print(f"\nTotal articles: {len(df)}")
print(f"Real articles: {(df['label'] == 1).sum()}")
print(f"Fake articles: {(df['label'] == 0).sum()}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")


Total articles: 44889
Real articles: 21417
Fake articles: 23472

Label distribution:
label
0    23472
1    21417
Name: count, dtype: int64


## Part 3: train/val/test (70/15/15, stratified)

In [11]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df['label']
)

# Second split: Split temp into 50/50 (15% val, 15% test)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['label']
)

print("FULL DATASET SPLIT (70/15/15)\n")

print(f"Training set:   {len(train_df):,}")
print(f"Validation set: {len(val_df):,}")
print(f"Test set:       {len(test_df):,}")
print(f"Total:          {len(df):,} samples")


FULL DATASET SPLIT (70/15/15)

Training set:   31,422
Validation set: 6,733
Test set:       6,734
Total:          44,889 samples


## Part 4: Prepare Data for BERT

In [15]:
# Get texts and labels as lists
train_texts = train_df['content'].tolist()
train_labels = train_df['label'].tolist()

val_texts = val_df['content'].tolist()
val_labels = val_df['label'].tolist()

test_texts = test_df['content'].tolist()
test_labels = test_df['label'].tolist()

print(f"Training texts prepared: {len(train_texts)}")
print(f"Validation texts prepared: {len(val_texts)}")
print(f"Test texts prepared: {len(test_texts)}")
print(f"\nTotal samples: {len(train_texts) + len(val_texts) + len(test_texts)}")
print(f"Sample text length: {len(train_texts[0])} characters")

Training texts prepared: 31422
Validation texts prepared: 6733
Test texts prepared: 6734

Total samples: 44889
Sample text length: 1215 characters


## Part 5: Train BERT Model

In [18]:
model, tokenizer = bert_utils.train_bert_model(
    train_texts,
    train_labels,
    model_name='bert-base-uncased',
    epochs=3,
    batch_size=16,
    learning_rate=2e-5
)

INFO:bert_utils:MacBook GPU (MPS) detected
INFO:bert_utils:Using device: mps
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO:bert_utils:Starting BERT training for 3 epochs...
Epoch 1/3: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 1964/1964 [10:20<00:00,  3.16batch/s, loss=0.0001]
INFO:bert_utils:Epoch 1/3 - Avg Loss: 0.0228
Epoch 2/3: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 1964/1964 [10:12<00:00,  3.21batch/s, loss=0.0001]
INFO:bert_utils:Epoch 2/3 - Avg Loss: 0.0054
Epoch 3/3: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 1964/1964 [10:09<00:00,  3.22batch/s, loss=0.0001]
IN

## Part 6: Evaluate on Test Set

In [21]:
val_results = bert_utils.evaluate_bert_model(
    model,
    tokenizer,
    val_texts,
    val_labels,
    batch_size=32
)
print(f"Accuracy:  {val_results['accuracy']:.4f}")
print(f"Precision: {val_results['precision']:.4f}")
print(f"Recall:    {val_results['recall']:.4f}")
print(f"F1 Score:  {val_results['f1']:.4f}")
print(f"ROC-AUC:   {val_results['roc_auc']:.4f}")

print("\nEvaluating on test set...\n")

test_results = bert_utils.evaluate_bert_model(
    model,
    tokenizer,
    test_texts,
    test_labels,
    batch_size=32
)

print(f"Accuracy:  {test_results['accuracy']:.4f}")
print(f"Precision: {test_results['precision']:.4f}")
print(f"Recall:    {test_results['recall']:.4f}")
print(f"F1 Score:  {test_results['f1']:.4f}")
print(f"ROC-AUC:   {test_results['roc_auc']:.4f}")

print(f"\nConfusion Matrix:")
cm = test_results['confusion_matrix']
print(f"  True Negatives:  {cm[0, 0]:5d}  (correctly identified fake)")
print(f"  False Positives: {cm[0, 1]:5d}  (real marked as fake)")
print(f"  False Negatives: {cm[1, 0]:5d}  (fake marked as real)")
print(f"  True Positives:  {cm[1, 1]:5d}  (correctly identified real)")


INFO:bert_utils:MacBook GPU (MPS) detected
INFO:bert_utils:Accuracy:  0.9984                                                                                                                               
INFO:bert_utils:Precision: 0.9972
INFO:bert_utils:Recall:    0.9994
INFO:bert_utils:F1 Score:  0.9983
INFO:bert_utils:ROC-AUC:   1.0000
INFO:bert_utils:MacBook GPU (MPS) detected


Accuracy:  0.9984
Precision: 0.9972
Recall:    0.9994
F1 Score:  0.9983
ROC-AUC:   1.0000

Evaluating on test set...



INFO:bert_utils:Accuracy:  0.9985                                                                                                                               
INFO:bert_utils:Precision: 0.9972
INFO:bert_utils:Recall:    0.9997
INFO:bert_utils:F1 Score:  0.9984
INFO:bert_utils:ROC-AUC:   1.0000


Accuracy:  0.9985
Precision: 0.9972
Recall:    0.9997
F1 Score:  0.9984
ROC-AUC:   1.0000

Confusion Matrix:
  True Negatives:   3512  (correctly identified fake)
  False Positives:     9  (real marked as fake)
  False Negatives:     1  (fake marked as real)
  True Positives:   3212  (correctly identified real)


## Part 7: Saving model

In [23]:
bert_utils.save_model(model, tokenizer, save_dir='models/bert_fake_news')


INFO:bert_utils:Model saved to models/bert_fake_news
